# Gouvernance multi-agents — voter, coalitionner, résister

Distillation du sous-projet EPITA `2.1.6_multiagent_governance_prototype`
(EPIC #4960, mandat Triple Distillation : porter l'essence vivante « sans le
bruit d'une année de régressions et d'itérations »).

**Position dans l'arc** — [Agentic-3-orchestration](Argument_Analysis_Agentic-3-orchestration.ipynb)
répond à « comment répartir le *travail* entre agents » (pipeline DAG,
orchestrateur conversationnel) ; ce notebook répond à « comment un collectif
*décide* » : sept méthodes de vote et de consensus, des personnalités d'agents,
des coalitions, et la question centrale de toute gouvernance — **que résiste
la décision à la manipulation ?**

L'organe `governance_methods.py` est un module **pur** (stdlib uniquement) :
les mécanismes sont calculables et testables sans aucune dépendance.

### Pourquoi la gouvernance est un problème dur

Répartir le travail entre agents (Agentic-3) se résout par de l'ingénierie ;
**décider à plusieurs** se heurte à un résultat d'impossibilité. Dès qu'un
collectif doit choisir parmi trois options ou plus à partir de préférences
individuelles ordonnées, le théorème d'Arrow (1951) interdit toute procédure
qui serait à la fois non-dictatoriale, unanime (si tous préfèrent X à Y, X
bat Y) et indépendante aux alternatives non pertinentes (le duel X vs Y ne
doit pas dépendre de la présence de Z). Il faut donc **choisir quelle
propriété sacrifier** — et chaque méthode de vote de ce notebook est un choix
différent de sacrifice. La suite ne débat pas de ces choix : elle les
**mesure** sur des scénarios où ils produisent des vainqueurs différents.

In [1]:
import sys
sys.path.insert(0, ".")  # organe governance_methods.py dans le dossier de la serie
import governance_methods as g
import json

counts = g.method_counts()
print(json.dumps(counts, ensure_ascii=False, indent=1))

{
 "methods": 7,
 "method_keys": [
  "borda",
  "byzantine",
  "condorcet",
  "majority",
  "plurality",
  "quadratic",
  "raft"
 ],
 "scenarios": 6,
 "scenario_names": [
  "byzantine_noise",
  "cyclic_majority",
  "dictatorship",
  "project_funding",
  "spoiler_candidate",
  "strategic_bloc"
 ]
}


### Lecture du compte

L'organe porte **7 méthodes** de vote/consensus et **6 scénarios étiquetés**
(distillés du banc de 16 du source — les 6 retenus couvrent chacun un phénomène
de choix collectif distinct : dictature, cycle, candidat spoiler, bloc
discipliné, bruit byzantin, arbitrage budgétaire). Les tests purs
(`tests/test_governance_methods.py`, 16 tests) figent ces comptes.

In [2]:
# Anatomie de deux scénarios : le cycle et le spoiler
for name in ["cyclic_majority", "spoiler_candidate"]:
    sc = g.SCENARIOS[name]
    print(f"--- {name} : {sc['description']}")
    for agent, perso, prefs in sc["agents"]:
        print(f"    {agent:<4} [{perso:<9}] {prefs}")
    print(f"    options : {sc['options']}")

--- cyclic_majority : Cycle de Condorcet : A bat B, B bat C, C bat A — aucun vainqueur net.
    V1   [stubborn ] ['A', 'B', 'C']
    V2   [stubborn ] ['B', 'C', 'A']
    V3   [stubborn ] ['C', 'A', 'B']
    options : ['A', 'B', 'C']
--- spoiler_candidate : Un candidat similaire divise un bloc : l'adversaire l'emporte.
    G1   [stubborn ] ['Gauche', 'Centre', 'Droite']
    G2   [stubborn ] ['Centre', 'Gauche', 'Droite']
    D1   [stubborn ] ['Droite', 'Centre', 'Gauche']
    options : ['Gauche', 'Centre', 'Droite']


### Lecture : préférences ordonnées et personnalités

Chaque agent porte une **préférence complète** (un ordre, pas un seul choix)
et une **personnalité** qui gouverne son vote sincère : `stubborn` vote son
premier choix quoi qu'il arrive, `flexible` peut suivre un indice de majorité,
`strategic` vote son second choix pour bloquer un adversaire, `random` tire au
sort. Toute la mécanique de vote se lit sur ces ordres : les méthodes de ce
notebook ne voient jamais la personnalité, elles ne voient que ce qu'elle
produit — des préférences déclarées. C'est exactement la surface d'attaque des
manipulations de fin de parcours : mentir sur ses préférences est
indistinguable d'avoir d'autres préférences.

In [3]:
# EFFET SPOILER : la même société, trois lectures du vainqueur
agents = g.build_agents("spoiler_candidate")
opts = g.SCENARIOS["spoiler_candidate"]["options"]

# Duels pairwise (coeur de Condorcet)
print("Duels :")
for i, o1 in enumerate(opts):
    for o2 in opts[i+1:]:
        v1 = sum(a.preferences.index(o1) < a.preferences.index(o2) for a in agents)
        v2 = len(agents) - v1
        print(f"  {o1} vs {o2} : {v1}-{v2}")

for m in ["majority", "borda", "condorcet"]:
    winner = g.simulate_vote(g.build_agents("spoiler_candidate"), opts, m, seed=0)["winner"]
    print(f"{m:<10} -> {winner}")

Duels :
  Gauche vs Centre : 1-2
  Gauche vs Droite : 2-1
  Centre vs Droite : 2-1
majority   -> Gauche
borda      -> Centre
condorcet  -> Centre


### Lecture : le spoiler mesuré

Les duels racontent tout : **Centre bat Gauche (2-1) et bat Droite (2-1)** —
Centre est le *vainqueur de Condorcet*, il gagne contre tout le monde. Pourtant
la **majorité à un tour élit Gauche** : G1 vote Gauche, G2 vote Centre, D1 vote
Droite — le bloc progressiste (2 électeurs sur 3) est divisé par la présence de
deux options voisines, et l'adversaire minoritaire l'emporte avec 1 voix sur 3.
Borda et Condorcet, qui lisent les préférences **complètes**, élisent Centre.
C'est l'effet spoiler — l'une des deux briques empiriques du théorème
d'impossibilité d'Arrow. Le décompte à un tour : Gauche 1 voix (G1), Centre
1 voix (G2), Droite 1 voix (D1) — trois voix, aucune majorité, et le tie-break
de `Counter.most_common` tranche pour la première insérée : l'égalité parfaite
à un tour est elle-même une convention silencieuse, invisible dans le résultat
affiché.

In [4]:
# CYCLE DE CONDORCET : A bat B, B bat C, C bat A
agents = g.build_agents("cyclic_majority")
opts = g.SCENARIOS["cyclic_majority"]["options"]
for i, o1 in enumerate(opts):
    for o2 in opts[i+1:]:
        v1 = sum(a.preferences.index(o1) < a.preferences.index(o2) for a in agents)
        print(f"  {o1} vs {o2} : {v1}-{len(agents)-v1}")
r = g.simulate_vote(agents, opts, "condorcet", seed=0)
print("vainqueur condorcet :", r["winner"], "(repli Borda)")

# Scores Borda explicites : l'egalite parfaite
scores = {o: 0 for o in opts}
for a in agents:
    for i, o in enumerate(a.preferences):
        scores[o] += len(opts) - i - 1
print("scores Borda :", scores)

  A vs B : 2-1
  A vs C : 1-2
  B vs C : 2-1
vainqueur condorcet : A (repli Borda)
scores Borda : {'A': 3, 'B': 3, 'C': 3}


### Lecture : le cycle et un repli qui ne résout rien

A bat B (2-1), B bat C (2-1), C bat A (2-1) : **cycle parfait**, aucun vainqueur
de Condorcet. Le repli Borda du source rend une **égalité parfaite 3 = 3 = 3** —
trois préférences strictes en rotation donnent exactement autant de points à
chacune. Le `A` final ne sort d'aucune préférence collective : il sort du
tie-break de `max()`, qui rend la **première clé d'insertion**. Le repli ne
résout pas le cycle, il le tranche par convention d'ordre — c'est la divergence
6, mesurée ici plutôt qu'affirmée (l'exercice 3 mesure ce que raft, qui tire son
leader, fait du même cycle).

In [5]:
# Tableau croise : le vainqueur selon la methode et le scenario
noms = counts["scenario_names"]
entete = f"{'scenario':<18}" + "".join(f"{m:<11}" for m in counts["method_keys"])
print(entete)
print("-" * len(entete))
for sc in noms:
    opts = g.SCENARIOS[sc]["options"]
    ligne = f"{sc:<18}"
    for m in counts["method_keys"]:
        w = g.simulate_vote(g.build_agents(sc), opts, m, seed=42)["winner"]
        ligne += f"{w:<11}"
    print(ligne)

scenario          borda      byzantine  condorcet  majority   plurality  quadratic  raft       
-----------------------------------------------------------------------------------------------
byzantine_noise   A          A          A          A          A          A          A          
cyclic_majority   A          A          A          A          A          A          C          
dictatorship      B          C          B          C          C          B          B          
project_funding   Ecole      Ecole      Ecole      Ecole      Ecole      Ecole      Ecole      
spoiler_candidate Centre     Gauche     Centre     Gauche     Gauche     Gauche     Centre     
strategic_bloc    A          A          A          A          A          A          A          


### Lecture : la méthode change le vainqueur

Sur six scénarios, trois rendent le même gagnant sous les sept méthodes
(`byzantine_noise`, `project_funding`, `strategic_bloc` : une dominance claire
absorbe le choix de procédure) et **trois divergent** :

- `dictatorship` : majorité/pluralité/byzantin élisent **C**, Borda/Condorcet/
  quadratique/Raft élisent **B** — le « dictateur » D1 (stratégique) vote son
  *second* choix, et la lecture complète des préférences préfère son sacrifice ;
- `spoiler_candidate` : majorité et dérivées élisent **Gauche**, les méthodes
  à préférences complètes élisent **Centre** ;
- `cyclic_majority` : seul **Raft déraille** (C) — son leader tiré au sort
  propose depuis n'importe quelle préférence.

Il n'existe pas de procédure neutre : changer la règle, c'est changer le
gagnant — sur un tiers de ce banc, sans toucher une seule préférence. Note de
lecture : les colonnes `majority` et `plurality` sont identiques par
construction (le source définit la pluralité comme un simple rappel de la
majorité — les tests purs figent cet alias), et la colonne `byzantine` à ratio
par défaut 0.2 suit la majorité sauf sur `dictatorship`, où son remplacement
du premier agent déplace exactement la voix qui faisait basculer C. Les trois
scénarios convergents montrent l'autre face : quand une dominance est claire
(bloc de 3 sur 5, option-Condorcet évidente), **toutes** les procédures
raisonnables l'expriment — la méthode ne devient décisive que près de
l'égalité, c'est-à-dire précisément là où elle est indécidable en principe.

### Mesurer une décision collective

Dire « méthode plus juste » n'a de sens qu'avec un instrument. L'organe en
fournit trois, complémentaires :

- **taux de consensus** : fraction des votes alignés sur le vainqueur — un
  collectif déchiré élit quelqu'un que peu ont voté ;
- **justice** : `1 - Gini(satisfactions)`, où le coefficient de Gini mesure
  l'inégalité d'une distribution (0 = tous égaux, 1 = maximalement inégal) —
  une méthode peut satisfaire beaucoup *en moyenne* en sacrifiant toujours
  les mêmes agents, le Gini le voit ;
- **satisfaction moyenne** : proximité du vainqueur au premier choix de
  chacun, normalisée par la longueur des préférences.

La cellule suivante croise ces instruments avec les méthodes : c'est là que
l'effet spoiler devient un chiffre, pas une intuition.

In [6]:
# Metriques : justice et satisfaction par methode (spoiler + dictatorship)
print(f"{'scenario':<12} {'methode':<10} {'consensus':<10} {'justice':<9} {'satisfaction'}")
for sc in ["spoiler_candidate", "dictatorship"]:
    opts = g.SCENARIOS[sc]["options"]
    for m in ["majority", "borda", "condorcet", "quadratic"]:
        r = g.simulate_vote(g.build_agents(sc), opts, m, seed=0)
        s = g.summarize(r)
        print(f"{sc:<12} {m:<10} {s['consensus_rate']:<10} {s['fairness']:<9} {s['satisfaction']}")

scenario     methode    consensus  justice   satisfaction
spoiler_candidate majority   0.333      0.556     0.5
spoiler_candidate borda      0.333      0.833     0.667
spoiler_candidate condorcet  0.333      0.833     0.667
spoiler_candidate quadratic  0.333      0.556     0.5
dictatorship majority   0.6        0.667     0.6
dictatorship borda      0.6        0.667     0.6
dictatorship condorcet  0.6        0.667     0.6
dictatorship quadratic  0.6        0.667     0.6


### Lecture : justice et satisfaction vont ensemble — ici

Sur le spoiler, les méthodes à préférences complètes dominent sur les deux axes :
justice (Gini) **0.833** contre **0.556**, satisfaction moyenne **0.667** contre
**0.5**. Élire le vainqueur de Condorcet n'est pas seulement « plus juste » au
sens de Gini : il satisfait *aussi* mieux le collectif — Centre est second
choix de ceux dont il bat le premier. Sur `dictatorship` en revanche, les quatre
méthodes rendent des métriques identiques (0.667 / 0.6) : ce scénario ne
discrimine pas les procédures, il montre surtout qu'un seul vote stratégique
déplace l'égalité initiale. Le banc garde son pouvoir discriminant précisément
sur `spoiler_candidate`.

### Et quand les agents tombent en panne ?

Les méthodes précédentes supposent des votes sincères et bien transmis. Un
système multi-agents réel affronte trois pannes : des agents **défaillants**
qui votent au hasard (le modèle byzantin), des agents **stratégiques** qui
déclarent autre chose que leur préférence vraie, et des **coalitions**
organisées qui coordonnent leurs votes. Le banc du source étiquetait
précisément ces axes — la suite les mesure un par un, en commençant par le
bruit : la cellule suivante fait croître la fraction d'agents byzantins et
observe le taux de consensus.

In [7]:
# ROBUSTESSE BYZANTINE : consensus moyen selon la fraction de votes aleatoires
# (10 seeds par palier, moyenne des taux de consensus)
opts = g.SCENARIOS["byzantine_noise"]["options"]
for ratio in [0.0, 0.2, 0.4]:
    taux = []
    for seed in range(10):
        agents = g.build_agents("byzantine_noise")
        r = g.simulate_vote(agents, opts, "byzantine",
                            context={"byzantine_ratio": ratio}, seed=seed)
        taux.append(g.consensus_rate(r))
    print(f"ratio byzantin {ratio:.1f} : consensus moyen = {sum(taux)/len(taux):.2f}")

ratio byzantin 0.0 : consensus moyen = 0.72
ratio byzantin 0.2 : consensus moyen = 0.76
ratio byzantin 0.4 : consensus moyen = 0.62


### Lecture : une courbe non monotone — l'artefact révélé

Intuition : plus de bruit byzantin = moins de consensus. La mesure dit
**0.72 → 0.76 → 0.62** : le palier 20 % de bruit est *meilleur* que le palier
0 %. Deux causes, toutes deux instructives :

1. **La base n'est pas silencieuse** : à ratio 0.0, les personnalités `random`
   (Z1, Z2) votent déjà aléatoirement — le « 0 % de bruit » du paramètre n'est
   pas le zéro de bruit du système.
2. **Le slicing déterministe (divergence 3)** : à 5 agents, ratio 0.2 = un
   byzantin = *toujours le premier agent* (H1), jamais un tirage. Le remplaçant
   aléatoire, sur ces 10 seeds, déplace le vainqueur vers un candidat
   mieux aligné des votes sincères — le taux remonte.

La dégradation réelle n'apparaît qu'à 40 % (0.62). C'est exactement le type
d'effet que la tranche `agents[:n]` du source rend invisible : un organe qui
tirait les byzantins au sort laverait cette bosse — au prix de perdre la
fidélité du port. Le port reste fidèle, la mesure reste honnête.

Le protocole lui-même est une leçon : **dix seeds par palier, moyenne rendue**
— un seul seed aurait rendu une des trois valeurs par palier et masqué la
non-monotonie ou l'aurait fabriquée par chance. Mesurer un système à tirage,
c'est mesurer une distribution, pas un point : le taux de consensus à ratio
0.2 sur UN seed prend n'importe quelle valeur de la distribution, et seule la
moyenne sur dix seeds dit quelque chose de la procédure.

In [8]:
# MANIPULATION : vote strategique et fausse coalition sur le bloc discipliné
opts = g.SCENARIOS["strategic_bloc"]["options"]

base = g.simulate_vote(g.build_agents("strategic_bloc"), opts, "majority", seed=0)
print(f"sincere            : vainqueur {base['winner']}, satisfaction {sum(base['satisfaction'])/len(base['satisfaction']):.2f}")

strat = g.apply_manipulation(g.build_agents("strategic_bloc"), "strategic")
r = g.simulate_vote(strat, opts, "majority", seed=0)
sat = sum(r['satisfaction'])/len(r['satisfaction'])
print(f"strategique        : vainqueur {r['winner']}, satisfaction {sat:.2f}")

false = g.apply_manipulation(g.build_agents("strategic_bloc"), "false_coalition", target="B")
r = g.simulate_vote(false, opts, "majority", seed=0)
sat = sum(r['satisfaction'])/len(r['satisfaction'])
print(f"fausse coalition B : vainqueur {r['winner']}, satisfaction {sat:.2f}")

sincere            : vainqueur A, satisfaction 0.60
strategique        : vainqueur B, satisfaction 0.70
fausse coalition B : vainqueur B, satisfaction 0.80


### Lecture : la manipulation rend le collectif... plus satisfait ?

Résultat contre-intuitif mesuré : sincère → A (satisfaction 0.60), vote
stratégique généralisé → B (0.70), fausse coalition pour B → B (0.80). Les deux
manipulations **augmentent** la satisfaction moyenne. L'explication est dans la
structure du scénario : le bloc A (3 agents) a B comme second choix, S1 a B
comme premier — B est le *compromis naturel* que le vote sincère ne trouve pas.
La manipulation, ici, pousse le collectif vers son compromis.

Le **piège de la moyenne** : la satisfaction du bloc passe de 1.0 (A gagne) à
0.5 (B gagne) — trois agents y perdent, deux y gagnent, et la moyenne monte.
Une gouvernance qui ne regarderait que la satisfaction moyenne peut consacrer
la manipulation ; la distribution (et le Gini de `summarize`) dit la vérité que
la moyenne tait. La contre-vérification est laissée à la cellule de
l'exercice 2 : croiser `summarize()` sincère contre manipulé montre la justice
baisser pendant que la satisfaction monte — les deux instruments divergent
précisément quand un seul d'entre eux suffit à tromper.

In [9]:
# COALITIONS : la confiance agrège des votes épars
agents = g.build_agents("dictatorship",
    trust={"O1": {"O2": 0.9, "O3": 0.5}, "O2": {"O1": 0.9}, "O3": {}, "O4": {"O3": 0.85}, "D1": {}})
coalitions = g.form_coalitions(agents)
for c in coalitions:
    print(f"{c[0].coalition_id} : {[a.name for a in c]} -> vote {c[0].preferences[0]} (leader {c[0].name})")

# Valeur de Shapley de la coalition O1-O2 : payoff = 1 si O1 present (B porte par O1)
def payoff(noms):
    return 1.0 if "O1" in noms else 0.0
vals = g.shapley_value(["O1", "O2"], payoff)
print("Shapley (payoff = B porte) :", {k: round(v, 2) for k, v in vals.items()})

coalition_1 : ['D1'] -> vote A (leader D1)
coalition_2 : ['O1', 'O2'] -> vote B (leader O1)
coalition_3 : ['O3'] -> vote B (leader O3)
coalition_4 : ['O4'] -> vote C (leader O4)
Shapley (payoff = B porte) : {'O1': 1.0, 'O2': 0.0}


### Lecture : la coalition exige une confiance lisible dans un sens précis

Deux coalitions se forment : O1↔O2 (confiance mutuelle 0.9) et... personne
d'autre. O4 fait pourtant confiance à O3 (0.85) — mais O3 ne truste personne :
la boucle du source parcourt les agents **en ordre de liste**, et O3 est
examiné *avant* O4 ; arrivé là, O3 ne propose à personne et reste seul. Quand
le tour d'O4 vient, O3 est déjà pris. Asymétrie mesurée de l'algorithme : la
coalition naît seulement si le **premier des deux, en ordre de liste**, truste
l'autre — une confiance unilatérale dans l'autre sens ne produit rien. Chaque
bloc vote ensuite la préférence de son *leader* (le premier rencontré), pas
celle de son membre le plus trusté.

La valeur de Shapley pose la question du **mérite** dans la coalition : avec
un payoff qui ne vaut que si O1 participe, O1 est pivotale dans les deux ordres
d'arrivée possibles et capte 1.0, O2 capte 0.0 — la coopération ne récompense
pas la présence, elle récompense la **contribution marginale**. C'est
l'instrument que le source appliquait aux coalitions de sa simulation ; ici il
est isolé, calculable à la main sur deux agents, et l'exercice 2 peut le
prolonger.

## Limites mesurées

Une distillation n'est fidèle que si elle dit ce qu'elle a corrigé en route :
chaque divergence ci-dessous a été **mesurée sur le source** avant d'être
écrite ici, pas supposée depuis une lecture de surface.

Ce port est **honnête sur ce qu'il ne fait pas** — six divergences mesurées sur
le source, pas supposées :

1. **La simulation du source n'appelait jamais ses méthodes de vote** :
   `simulate_governance` assignait `method_fn = GOVERNANCE_METHODS[method]`
   puis déroulait un tally de blocs sans jamais l'invoquer — les 7 méthodes y
   étaient du code mort. L'organe corrige : `simulate_vote` applique la méthode.
2. **Le « vote quadratique » du source n'est pas quadratique** : aucun coût en
   somme de carrés ; un agent flexible coupe juste son budget en deux. Porté
   tel quel — l'exercice 1 fait implémenter le vrai.
3. **Les byzantins sont les n premiers agents** (tranche déterministe
   `agents[:n]`), pas un tirage : mesuré, porté fidèlement.
4. **La voie mémoire du source référence `options` hors de portée** (NameError
   latent dès que le contexte ne porte pas `"options"`) : non portée, la
   satisfaction est recalculée depuis les préférences.
5. **La médiation à probabilités fixes** (0.8/0.5/0.7 codés en dur) n'est pas
   portée : seul `detect_conflicts` l'est.
6. **Le repli Borda en cas de cycle** conserve le tie-break de `max()` (première
   clé d'insertion) — démontré sur le cycle ci-dessus où Borda rend 3=3=3.

Aussi : les méthodes aléatoires (byzantin, raft, personnalité `random`) sont
**seedées** — les sorties committées sont reproductibles à l'identique, mais un
autre seed peut rendre un autre vainqueur (l'exercice 3 le mesure).

### Exercice 1 — le VRAI vote quadratique

Le source annonce un coût quadratique mais ne l'implémente pas (limite 2).
Implémentez-le pour de vrai : chaque agent dispose d'un budget de crédits, le
**coût** payé pour acheter `v` voix sur une option est `v²` (le budget
s'épuise en carrés), et l'allocation optimale d'un agent qui aime une option
d'intensité `i` est d'y mettre `floor(sqrt(budget * i))` voix. Faites voter le
scénario `project_funding` : le vainqueur change-t-il par rapport au port
« budget coupé en deux » de l'organe ?

In [10]:
def vrai_quadratique(agents, options, budget=9, intensite=None):
    # Etape 1 : pour chaque agent, voix par option = floor(sqrt(budget * intensite(option)))
    # Etape 2 : cumuler les voix par option sur tous les agents
    # Etape 3 : rendre l'option au total max
    # Indice : intensite = 1.0 premier choix, 0.5 deuxieme, 0.0 ensuite ;
    # tout le budget sur le premier choix achete floor(sqrt(9)) = 3 voix.
    return None  # TODO etudiant


print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 — construire la société où majorité et Borda se contredisent

Construisez un scénario (dans le format des tuples de `SCENARIOS`) où la
majorité à un tour et Borda **ne désignent pas le même vainqueur**, puis
vérifiez ce que dit Condorcet. Mesurez `summarize()` pour les deux méthodes :
la plus juste (justice Gini) est-elle celle qui satisfait le plus d'agents ?

In [11]:
ma_societe = {
    "options": [],
    "agents": [
        # ("nom", "personnalite", [preferences completes])
    ],
}
# Indice : spoiler_candidate est deja un cas majority != borda ; cherchez-en un
# AUTRE (4 options ou 7 agents), puis g.summarize(g.simulate_vote(...)) x 2.
print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 — stabilité multi-seed et témoins négatifs

(a) `raft` tire son leader au sort : mesurez sa **stabilité** sur le cycle de
Condorcet — 10 seeds, comptez la distribution des vainqueurs. (b) Témoin
négatif : sur `strategic_bloc` (un bloc de 3 sur 5), toutes les méthodes et
tous les seeds doivent rendre le même vainqueur — vérifiez qu'aucun seed ne
déraille.

In [12]:
from collections import Counter
gagnants = Counter()
# Etape 1 : boucle sur 10 seeds -> simulate_vote(..., "raft", seed=s) sur cyclic_majority
# Etape 2 : gagnants[winner] += 1, print(gagnants)
# Etape 3 : temoin negatif sur strategic_bloc (meme distribution, meme gagnant partout)
# Indice : g.stability([...]) rend 1 si un seul vainqueur dans la liste, 0 sinon.
resultat_attendu = None  # TODO etudiant
print("Exercice 3 a completer")

Exercice 3 a completer


## Conclusion

Ce notebook a fait mesurer, sur des préférences identiques, ce que le théorème
d'Arrow (1951) énonce en général : **aucune procédure de agrégation de trois
options ou plus ne peut être simultanément** non-dictatoriale, unanime et
indépendante aux alternatives non pertinentes. Le spoiler et le cycle sont ses
deux manifestations mesurées ici — pas des accidents de scénario, des
propriétés structurelles du choix collectif.

- La **même société de préférences** rend des vainqueurs différents selon la
  méthode : il n'existe pas de procédure de vote neutre (flèche du théorème
  d'Arrow, dont le cycle et le spoiler ci-dessus sont les deux briques
  empiriques).
- Les **métriques** (consensus, justice de Gini, satisfaction) rendent ces
  divergences mesurables au lieu de les débattre.
- La gouvernance se teste comme un système : **bruit byzantin, manipulation,
  stabilité multi-seed** — exactement les axes que le banc de scénarios du
  source étiquetait.

**Arc** : Agentic-3 (orchestrer le travail) → ce notebook (décider à
plusieurs) → Agentic-4 (capstone). L'organe `governance_methods.py` et ses 16
tests purs sont réutilisables pour vos propres scénarios : construire une
société, la faire voter sous les sept méthodes, et lire ce que chacune
sacrifie — c'est tout l'appareil qu'il faut pour cela.